In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
TRAIN_PATH = '/content/drive/MyDrive/office_products_split/Office_Products.train.csv'
VALID_PATH = '/content/drive/MyDrive/office_products_split/Office_Products.valid.csv'
TEST_PATH = '/content/drive/MyDrive/office_products_split/Office_Products.test.csv'

EMBEDDING_PATH = '/content/drive/My Drive/RS/feature_meta_Sbert/item_embedding.parquet'

In [ ]:
!head -20 $TRAIN_PATH

user_id,parent_asin,rating,timestamp,history
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B079HQK8HG,5.0,1534443278981,
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B0BNNDBH81,5.0,1534444231358,B079HQK8HG
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B0C3R84FLG,1.0,1561156591217,B079HQK8HG B0BNNDBH81
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B098K24779,5.0,1589933892398,B079HQK8HG B0BNNDBH81 B0C3R84FLG
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07ZPB8T4P,1.0,1611642164889,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08CGCWQM3,5.0,1619294576562,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779 B07ZPB8T4P
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07DP8BYSY,3.0,1622449935071,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779 B07ZPB8T4P B08CGCWQM3
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07664G5SL,5.0,1622450316093,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779 B07ZPB8T4P B08CGCWQM3 B07DP8BYSY
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07NRD674H,1.0,1623382142051,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779 B07ZPB8T4P B08CGCWQM3 B07DP8BYSY B07664G5SL
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B0C89HT

In [ ]:
!head -20 $VALID_PATH

user_id,parent_asin,rating,timestamp,history
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B0B5RTTZ95,3.0,1642564061814,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779 B07ZPB8T4P B08CGCWQM3 B07DP8BYSY B07664G5SL B07NRD674H B0C89HT5G5 B07NWP25SX B08XM9Y46P B07T4NX5F6
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B06XY7YNHG,5.0,1642733588149,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779 B07ZPB8T4P B08CGCWQM3 B07DP8BYSY B07664G5SL B07NRD674H B0C89HT5G5 B07NWP25SX B08XM9Y46P B07T4NX5F6 B0B5RTTZ95
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B09KLJ9NJH,4.0,1642826493637,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779 B07ZPB8T4P B08CGCWQM3 B07DP8BYSY B07664G5SL B07NRD674H B0C89HT5G5 B07NWP25SX B08XM9Y46P B07T4NX5F6 B0B5RTTZ95 B06XY7YNHG
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08L4K4QKR,3.0,1643160661334,B079HQK8HG B0BNNDBH81 B0C3R84FLG B098K24779 B07ZPB8T4P B08CGCWQM3 B07DP8BYSY B07664G5SL B07NRD674H B0C89HT5G5 B07NWP25SX B08XM9Y46P B07T4NX5F6 B0B5RTTZ95 B06XY7YNHG B09KLJ9NJH
AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B00YL5GBWG,4.0,1643161893294,B079HQK8HG B0BNNDBH81 B0C3R



## Build vector db

In [34]:
!pip install faiss-cpu

In [35]:
import pandas as pd
import numpy as np


def prepare_embeddings(df, embedding_col="embedding", id_col="product_id"):
    ids = df[id_col].astype(str).tolist()
    xb = np.vstack(df[embedding_col].values).astype("float32")
    return ids, xb

In [36]:
import faiss
import numpy as np
import pickle
import os

def build_hnsw_index(
    df,
    embedding_col="embedding",
    id_col="product_id",
    M=32,
    ef_construction=200,
    ef_search=256,
    normalize=True
):
    ids, xb = prepare_embeddings(df, embedding_col, id_col)
    dim = xb.shape[1]

    if normalize:
        faiss.normalize_L2(xb)  # để cosine similarity ~= inner product

    index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = ef_construction
    index.hnsw.efSearch = ef_search

    index.add(xb)

    return index, ids

In [37]:
def search_topk(index, ids, query_embedding, topk=200, normalize=True):
    xq = np.array(query_embedding, dtype="float32").reshape(1, -1)

    if normalize:
        faiss.normalize_L2(xq)

    scores, idx = index.search(xq, topk)

    results = []
    for score, i in zip(scores[0], idx[0]):
        if i == -1:
            continue
        results.append({
            "product_id": ids[i],
            "score": float(score)
        })
    return results

In [38]:
def save_faiss_index(index, ids, index_path="faiss_hnsw.index", meta_path="faiss_ids.pkl"):
    faiss.write_index(index, index_path)
    with open(meta_path, "wb") as f:
        pickle.dump(ids, f)

def load_faiss_index(index_path="faiss_hnsw.index", meta_path="faiss_ids.pkl"):
    index = faiss.read_index(index_path)
    with open(meta_path, "rb") as f:
        ids = pickle.load(f)
    return index, ids

In [ ]:
embedding_df = pd.read_parquet(EMBEDDING_PATH)


In [ ]:
index, ids = build_hnsw_index(
    embedding_df,
    embedding_col="embedding",
    id_col="product_id",
    M=32,
    ef_construction=200,
    ef_search=256,
    normalize=True
)

save_faiss_index(index, ids, "products_hnsw.index", "products_ids.pkl")

query_vec = embedding_df.iloc[0]["embedding"]   # ví dụ lấy embedding của 1 item làm query
top200 = search_topk(index, ids, query_vec, topk=200, normalize=True)

print(top200[:10])

[{'product_id': '0000306037', 'score': 1.0}, {'product_id': 'B00GYFTVE6', 'score': 0.9999999403953552}, {'product_id': 'B00GYG01PI', 'score': 0.9421308636665344}, {'product_id': '0000306045', 'score': 0.941230833530426}, {'product_id': 'B00GYFVU5Y', 'score': 0.941230833530426}, {'product_id': '0000306010', 'score': 0.9407011866569519}, {'product_id': 'B00GYG2JSA', 'score': 0.9264885187149048}, {'product_id': 'B00I8VX1S6', 'score': 0.9191873073577881}, {'product_id': '0000306096', 'score': 0.9042991399765015}, {'product_id': 'B00GWQXSG4', 'score': 0.9036721587181091}]


In [ ]:
from pprint import pprint

pprint(top200[:10])

[{'product_id': '0000306037', 'score': 1.0},
 {'product_id': 'B00GYFTVE6', 'score': 0.9999999403953552},
 {'product_id': 'B00GYG01PI', 'score': 0.9421308636665344},
 {'product_id': '0000306045', 'score': 0.941230833530426},
 {'product_id': 'B00GYFVU5Y', 'score': 0.941230833530426},
 {'product_id': '0000306010', 'score': 0.9407011866569519},
 {'product_id': 'B00GYG2JSA', 'score': 0.9264885187149048},
 {'product_id': 'B00I8VX1S6', 'score': 0.9191873073577881},
 {'product_id': '0000306096', 'score': 0.9042991399765015},
 {'product_id': 'B00GWQXSG4', 'score': 0.9036721587181091}]


## Load vector db

In [39]:
INDEX_PATH = '/content/drive/MyDrive/FAISS/products_hnsw.index'
IDS_PATH = '/content/drive/MyDrive/FAISS/products_ids.pkl'

index, ids = load_faiss_index(INDEX_PATH, IDS_PATH)
print(f"Loaded index with {index.ntotal} vectors.")
print(f"Loaded {len(ids)} product IDs.")

Loaded index with 710503 vectors.
Loaded 710503 product IDs.


In [40]:
EMBED_MATRIX_DIR = "/content/drive/MyDrive/office_products_split/embedding_matrix"

emb_matrix = np.load(f"{EMBED_MATRIX_DIR}/embeddings.npy", mmap_mode="r")
with open(f"{EMBED_MATRIX_DIR}/id2idx.pkl", "rb") as f:
    id2idx = pickle.load(f)

## Baseline weighted avg

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
train_df['history'] = train_df['history'].apply(lambda x: x.split() if isinstance(x, str) else [])

valid_df = pd.read_csv(VALID_PATH)
valid_df['history'] = valid_df['history'].apply(lambda x: x.split() if isinstance(x, str) else [])

test_df = pd.read_csv(TEST_PATH)
test_df['history'] = test_df['history'].apply(lambda x: x.split() if isinstance(x, str) else [])

In [ ]:
import gc

train_df["rating"] = train_df["rating"].astype(int)

agg_df = (
    train_df.groupby("user_id", as_index=False)
    .agg(
        history=("history", lambda x: list(x.iloc[-1])),
        last_item=("parent_asin", "last"),
        ratings=("rating", list),
    )
)

agg_df["history"] = agg_df["history"] + agg_df["last_item"].map(lambda x: [x])
agg_df = agg_df.drop(columns=["last_item"])

del train_df
gc.collect()

agg_df.head()
agg_df.to_parquet("/content/drive/MyDrive/office_products_split/agg_train.parquet", index=False)

In [ ]:
valid_df["rating"] = valid_df["rating"].astype(int)

agg_valid_df = (
    valid_df.groupby("user_id", as_index=False)
    .agg(
        history=("history", lambda x: list(x.iloc[-1])),
        last_item=("parent_asin", "last"),
        ratings=("rating", list),
    )
)

agg_valid_df["history"] = agg_valid_df["history"] + agg_valid_df["last_item"].map(lambda x: [x])
agg_valid_df = agg_valid_df.drop(columns=["last_item"])

merged_df = agg_df.merge(agg_valid_df, on="user_id", how="outer", suffixes=("_train", "_valid"))

merged_df["history"] = merged_df.apply(
    lambda row: row["history_valid"] if isinstance(row["history_valid"], list)
    else (row["history_train"] if isinstance(row["history_train"], list) else []),
    axis=1,
)

merged_df["ratings"] = merged_df.apply(
    lambda row: (row["ratings_train"] if isinstance(row["ratings_train"], list) else [])
    + (row["ratings_valid"] if isinstance(row["ratings_valid"], list) else []),
    axis=1,
)

agg_df = merged_df[["user_id", "history", "ratings"]]

del valid_df, agg_valid_df, merged_df
gc.collect()

agg_df.head()
agg_df.to_parquet("/content/drive/MyDrive/office_products_split/agg_train_valid.parquet", index=False)

In [ ]:
agg_df.head()

,user_id,history,ratings
0,AE22222X4BO3JQVOMEHHRPVRTB6A,[B08WCDLKFK],[1]
1,AE2222FRPDMNOMYOMCWIANTXP7UQ,"[B01LYCT9GO, B01IMRJ3PU, B0725JJHKX]","[5, 5, 5]"
2,AE2222VGHFD37G7NOCWXASYY3ZDQ,[B07D8M41Y1],[3]
3,AE2222WEGI5MUXLFZW6UKKUSI74A,[B00SYDAS8Y],[5]
4,AE22236AFRRSMQIKGG7TPTB75QEA,"[B085SCZZSS, B0BCYRD6DW]","[4, 4]"


In [10]:
agg_df = pd.read_parquet("/content/drive/MyDrive/office_products_split/agg_train_valid.parquet")

In [41]:
test_df = pd.read_csv(TEST_PATH)
test_df['history'] = test_df['history'].apply(lambda x: x.split() if isinstance(x, str) else [])

test_df.head()

,user_id,parent_asin,rating,timestamp,history
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B097SFY5ZS,3.0,1659799390978,"[B079HQK8HG, B0BNNDBH81, B0C3R84FLG, B098K2477..."
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07BR2PBJN,4.0,1659806066713,"[B079HQK8HG, B0BNNDBH81, B0C3R84FLG, B098K2477..."
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07JDZ5J46,1.0,1660188831933,"[B079HQK8HG, B0BNNDBH81, B0C3R84FLG, B098K2477..."
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08L6H23JZ,4.0,1677939160682,"[B079HQK8HG, B0BNNDBH81, B0C3R84FLG, B098K2477..."
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B01MZ3SD2X,5.0,1677939345945,"[B079HQK8HG, B0BNNDBH81, B0C3R84FLG, B098K2477..."


In [42]:
positive_test_df = test_df[test_df.rating >= 4.0]


In [29]:
sample = positive_test_df.sample(1000, random_state=42).iloc[44]

user_id = sample["user_id"]
target_item = sample["parent_asin"]
target_ts = sample["timestamp"]

base_row = agg_df[agg_df["user_id"] == user_id]
base_history = list(base_row.iloc[0]["history"]) if len(base_row) > 0 else []
base_ratings = [int(r) for r in base_row.iloc[0]["ratings"]] if len(base_row) > 0 else []

prev_test = (
    test_df[(test_df["user_id"] == user_id) & (test_df["timestamp"] < target_ts)]
    .sort_values("timestamp")
)

full_history = base_history + prev_test["parent_asin"].tolist()
full_ratings = base_ratings + prev_test["rating"].astype(int).tolist()

hist_embs, hist_weights = [], []
for item, r in zip(full_history, full_ratings):
    idx = id2idx.get(item)
    if idx is not None:
        hist_embs.append(emb_matrix[idx])
        hist_weights.append(float(r))

hist_embs = np.vstack(hist_embs).astype(np.float32)
hist_weights = np.asarray(hist_weights, dtype=np.float32)
user_profile = np.average(hist_embs, axis=0, weights=hist_weights).astype(np.float32)

target_idx = id2idx[target_item]
target_emb = np.asarray(emb_matrix[target_idx], dtype=np.float32)

user_profile_norm = user_profile.reshape(1, -1).copy()
target_emb_norm = target_emb.reshape(1, -1).copy()
faiss.normalize_L2(user_profile_norm)
faiss.normalize_L2(target_emb_norm)
target_score = float(np.dot(user_profile_norm[0], target_emb_norm[0]))

retrieved = search_topk(index, ids, user_profile, topk=300, normalize=True)
seen = set(full_history)
top100 = [x for x in retrieved if x["product_id"] not in seen][:100]

pd.DataFrame(top100[:10]), target_score

(   product_id     score
 0  B0BNBZGSPR  0.787384
 1  B0BM42XLHZ  0.773764
 2  B0BX8R3JTM  0.773543
 3  B0BY2BC1F1  0.772479
 4  B07V3ZJTQ4  0.772157
 5  B0BYD9WZZ1  0.770420
 6  B0BR64M14Y  0.769027
 7  B08CXRJ62D  0.767161
 8  B08FSNBB9M  0.764801
 9  B07SC9XS3M  0.764760,
 0.5519500970840454)

In [30]:
list(zip(full_history, full_ratings))

[('B0BK915FGG', 5), ('B0BDY8QW9S', 4), ('B0BGXN141N', 5), ('B0BJ2SHCBV', 1)]

In [33]:
target_item


'B0BNN8R8FC'

## Negative Profile

In [43]:
import numpy as np


def search_topk_exact_negative(
    ids,
    item_embeddings,
    user_pos_embedding,
    user_neg_embedding,
    lamb=0.2,
    topk=200,
):
    xb = np.asarray(item_embeddings, dtype="float32")
    u_pos = np.asarray(user_pos_embedding, dtype="float32").reshape(1, -1)
    u_neg = np.asarray(user_neg_embedding, dtype="float32").reshape(1, -1)

    # normalize item embeddings
    xb_norm = np.linalg.norm(xb, axis=1, keepdims=True)
    xb_norm = np.maximum(xb_norm, 1e-12)
    xb = xb / xb_norm

    # normalize user profiles
    u_pos = u_pos / max(np.linalg.norm(u_pos), 1e-12)
    u_neg = u_neg / max(np.linalg.norm(u_neg), 1e-12)

    # exact scores on all items
    pos_scores = (xb @ u_pos.T).reshape(-1)   # cos(u_pos, x)
    neg_scores = (xb @ u_neg.T).reshape(-1)   # cos(u_neg, x)
    final_scores = pos_scores - lamb * neg_scores

    # exact top-k
    topk = min(topk, len(ids))
    top_idx = np.argpartition(-final_scores, topk - 1)[:topk]
    top_idx = top_idx[np.argsort(-final_scores[top_idx])]

    results = []
    for i in top_idx:
        results.append({
            "product_id": ids[i],
            "score": float(final_scores[i]),
            "pos_score": float(pos_scores[i]),
            "neg_score": float(neg_scores[i]),
        })

    return results

In [44]:
def build_user_profiles(history, ratings, id2idx, emb_matrix):
    pos_embs, pos_weights = [], []
    neg_embs, neg_weights = [], []

    for item, r in zip(history, ratings):
        idx = id2idx.get(item)
        if idx is None:
            continue

        emb = emb_matrix[idx]
        if r >= 4:
            pos_embs.append(emb)
            pos_weights.append(float(r))
        elif r <= 2:
            neg_embs.append(emb)
            neg_weights.append(float(r))

    if len(pos_embs) > 0:
        u_pos = np.average(pos_embs, axis=0, weights=pos_weights).astype(np.float32)
    else:
        u_pos = np.zeros(emb_matrix.shape[1], dtype=np.float32)

    if len(neg_embs) > 0:
        u_neg = np.average(neg_embs, axis=0, weights=neg_weights).astype(np.float32)
    else:
        u_neg = np.zeros(emb_matrix.shape[1], dtype=np.float32)

    return u_pos, u_neg

In [34]:
u_pos, u_neg = build_user_profiles(full_history, full_ratings, id2idx, emb_matrix)

print(f"User ID: {user_id}")
print(f"Target Item: {target_item}")
print(f"History length: {len(full_history)}")
print(f"Negative profile exists: {not np.all(u_neg == 0)}")

target_idx = id2idx[target_item]
target_vec = emb_matrix[target_idx].reshape(1, -1).astype("float32")

target_vec = target_vec / np.maximum(np.linalg.norm(target_vec), 1e-12)
u_pos_norm = u_pos / np.maximum(np.linalg.norm(u_pos), 1e-12)
u_neg_norm = u_neg / np.maximum(np.linalg.norm(u_neg), 1e-12)

lamb = 0.3
pos_score_t = float(target_vec @ u_pos_norm.T)
neg_score_t = float(target_vec @ u_neg_norm.T)
target_final_score = pos_score_t - lamb * neg_score_t

results_with_neg = search_topk_exact_negative(
    ids=ids,
    item_embeddings=emb_matrix,
    user_pos_embedding=u_pos,
    user_neg_embedding=u_neg,
    lamb=lamb,
    topk=200
)

seen = set(full_history)
final_recs = [x for x in results_with_neg if x['product_id'] not in seen][:10]

print(f"\nTarget Item Final Score: {target_final_score:.4f} (Pos: {pos_score_t:.4f}, Neg: {neg_score_t:.4f})")
print("\nTop 10 Recommendations:")
display(pd.DataFrame(final_recs))

User ID: AGHTBEMSA645ESLIY446N5TQ465Q
Target Item: B0BNN8R8FC
History length: 4
Negative profile exists: True


/tmp/ipykernel_15290/1021660394.py:16: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pos_score_t = float(target_vec @ u_pos_norm.T)
/tmp/ipykernel_15290/1021660394.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  neg_score_t = float(target_vec @ u_neg_norm.T)



Target Item Final Score: 0.5127 (Pos: 0.5561, Neg: 0.1447)

Top 10 Recommendations:


,product_id,score,pos_score,neg_score
0,B00XCHK7H4,0.722531,0.769143,0.155370
1,B0BNBZGSPR,0.706242,0.786961,0.269063
2,B0BY2BC1F1,0.701302,0.774174,0.242907
3,B09H2VXKG8,0.691473,0.752806,0.204445
4,B0BN2NZ22J,0.687965,0.760196,0.240771
5,B0BM42XLHZ,0.686132,0.771372,0.284132
6,B091SZY6MD,0.684787,0.747774,0.209956
7,B07SC9XS3M,0.684014,0.763865,0.266170
8,B0C3L5HYSH,0.683502,0.754415,0.236377
9,B0BYD9WZZ1,0.682287,0.767817,0.285101


## Attention - based

### Soft labeled for week-supervision

In [15]:
def count_pos_neg(ratings):
    pos = sum(1 for r in ratings if r >= 4)
    neg = sum(1 for r in ratings if r <= 2)
    last_is_pos = (ratings[-1] >= 4) if len(ratings) > 0 else False
    return pos, neg, last_is_pos

stats_list = agg_df['ratings'].apply(count_pos_neg)
agg_df['num_pos'] = [x[0] for x in stats_list]
agg_df['num_neg'] = [x[1] for x in stats_list]
agg_df['last_item_pos'] = [x[2] for x in stats_list]

filtered_users = agg_df[
    (agg_df['num_pos'] >= 2) &
    (agg_df['num_neg'] >= 1) &
    (agg_df['last_item_pos'] == True)
].copy()

print(f"Tổng số user ban đầu: {len(agg_df)}")
print(f"Số lượng user thỏa điều kiện (>=2 pos, >=1 neg, last item pos): {len(filtered_users)}")

print("\n--- Thống kê trên tập user đã lọc ---")
display(filtered_users[['num_pos', 'num_neg']].describe())

filtered_users['hist_len'] = filtered_users['history'].apply(len)
print(f"Độ dài lịch sử trung bình: {filtered_users['hist_len'].mean():.2f}")
print(f"Độ dài lịch sử lớn nhất: {filtered_users['hist_len'].max()}")

print("\nVí dụ 5 user đầu tiên:")
filtered_users.head()

Tổng số user ban đầu: 6889243
Số lượng user thỏa điều kiện (>=2 pos, >=1 neg, last item pos): 134199

--- Thống kê trên tập user đã lọc ---


,num_pos,num_neg
count,134199.000000,134199.000000
mean,4.750818,1.290949
std,6.809547,0.820382
min,2.000000,1.000000
25%,2.000000,1.000000
50%,3.000000,1.000000
75%,5.000000,1.000000
max,680.000000,61.000000


Độ dài lịch sử trung bình: 6.36
Độ dài lịch sử lớn nhất: 760

Ví dụ 5 user đầu tiên:


,user_id,history,ratings,num_pos,num_neg,last_item_pos,hist_len
51,AE2233C6IOCM5BYPGQJWQN2U3DPA,"[B00NDFY19K, B009PPF0ZG, B01ALJICLS]","[4, 1, 5]",2,1,True,3
55,AE2235Q53V246ISFXCSESYHAVNUA,"[B0BCYZ8FN7, B07C85RLS2, B07CWHFB45, B002OHDTM...","[5, 5, 1, 3, 4]",3,1,True,5
104,AE2244ILMBLRPTIN7VW7YDKRI2YA,"[B00MBB2V7Q, B07Z4B264R, B09XJHL3QL]","[5, 1, 4]",2,1,True,3
128,AE224GVO7OHTYF26U6ER6BEVIUAQ,"[B00KY4QS3K, B00006IEI4, B00MC52KFY, B08DDF93F...","[3, 5, 3, 2, 5, 5, 5]",4,1,True,7
129,AE224HM2QAW5TTSDL2QRDERA6KMA,"[B00000JBLQ, B00004S7P0, B00008VF1V, B00008Y2E...","[5, 5, 2, 4, 5, 5]",5,1,True,6


In [16]:
import random

def split_user_data(row):
    hist = list(row['history'])
    rats = list(row['ratings'])

    pos_target = hist.pop()
    pos_target_rat = rats.pop()

    neg_indices = [i for i, r in enumerate(rats) if r <= 2]

    selected_neg_idx = random.choice(neg_indices)

    neg_target = hist.pop(selected_neg_idx)
    neg_target_rat = rats.pop(selected_neg_idx)

    return pd.Series({
        'history': hist,
        'ratings': rats,
        'positive_target': pos_target,
        'positive_target_rating': pos_target_rat,
        'negative_target': neg_target,
        'negative_target_rating': neg_target_rat
    })

processed_df = filtered_users.apply(split_user_data, axis=1)

final_samples_df = pd.concat([filtered_users[['user_id']], processed_df], axis=1)

print(f"Đã xử lý xong {len(final_samples_df)} samples.")
print("\nCấu trúc dữ liệu sau khi tách:")
display(final_samples_df.head())

sample_row = final_samples_df.iloc[0]
print(f"\nKiểm tra User {sample_row['user_id']}:")
print(f"- Positive Target: {sample_row['positive_target']} (Rating: {sample_row['positive_target_rating']})")
print(f"- Negative Target: {sample_row['negative_target']} (Rating: {sample_row['negative_target_rating']})")
print(f"- History còn lại: {len(sample_row['history'])} items")

Đã xử lý xong 134199 samples.

Cấu trúc dữ liệu sau khi tách:


,user_id,history,ratings,positive_target,positive_target_rating,negative_target,negative_target_rating
51,AE2233C6IOCM5BYPGQJWQN2U3DPA,[B00NDFY19K],[4],B01ALJICLS,5,B009PPF0ZG,1
55,AE2235Q53V246ISFXCSESYHAVNUA,"[B0BCYZ8FN7, B07C85RLS2, B002OHDTMI]","[5, 5, 3]",B073RR8LFY,4,B07CWHFB45,1
104,AE2244ILMBLRPTIN7VW7YDKRI2YA,[B00MBB2V7Q],[5],B09XJHL3QL,4,B07Z4B264R,1
128,AE224GVO7OHTYF26U6ER6BEVIUAQ,"[B00KY4QS3K, B00006IEI4, B00MC52KFY, B0B1F2465...","[3, 5, 3, 5, 5]",B07589CD62,5,B08DDF93FX,2
129,AE224HM2QAW5TTSDL2QRDERA6KMA,"[B00000JBLQ, B00004S7P0, B00008Y2EK, B00006B7P4]","[5, 5, 4, 5]",B00007L6C2,5,B00008VF1V,2



Kiểm tra User AE2233C6IOCM5BYPGQJWQN2U3DPA:
- Positive Target: B01ALJICLS (Rating: 5)
- Negative Target: B009PPF0ZG (Rating: 1)
- History còn lại: 1 items


In [21]:
def to_native_types(val):
    if isinstance(val, list):
        return [getattr(x, "item", lambda: x)() if hasattr(x, "item") else x for x in val]
    return val

final_samples_df['history'] = final_samples_df['history'].apply(to_native_types)
final_samples_df['ratings'] = final_samples_df['ratings'].apply(to_native_types)

In [22]:
final_samples_df.to_csv("/content/drive/MyDrive/office_products_split/negative_feedback_train.csv", index=False)

### Baseline with negative feedback

In [45]:
import ast

neg_feedback_df = pd.read_csv("/content/drive/MyDrive/office_products_split/negative_feedback_train.csv")

neg_feedback_df['history'] = neg_feedback_df['history'].apply(ast.literal_eval)
neg_feedback_df['ratings'] = neg_feedback_df['ratings'].apply(ast.literal_eval)

neg_feedback_df = neg_feedback_df[neg_feedback_df['history'].apply(len) >= 2]

print(f"Shape sau khi lọc: {neg_feedback_df.shape}")
neg_feedback_df.head()

Shape sau khi lọc: (93557, 7)


,user_id,history,ratings,positive_target,positive_target_rating,negative_target,negative_target_rating
1,AE2235Q53V246ISFXCSESYHAVNUA,"[B0BCYZ8FN7, B07C85RLS2, B002OHDTMI]","[5, 5, 3]",B073RR8LFY,4,B07CWHFB45,1
3,AE224GVO7OHTYF26U6ER6BEVIUAQ,"[B00KY4QS3K, B00006IEI4, B00MC52KFY, B0B1F2465...","[3, 5, 3, 5, 5]",B07589CD62,5,B08DDF93FX,2
4,AE224HM2QAW5TTSDL2QRDERA6KMA,"[B00000JBLQ, B00004S7P0, B00008Y2EK, B00006B7P4]","[5, 5, 4, 5]",B00007L6C2,5,B00008VF1V,2
5,AE22AR6B4VOCUGJALEIIHYV6546Q,"[B00BF5LREM, B013I2XCGK, B014C7DH56]","[5, 4, 5]",B08LDZB1C4,5,B010B03Y1U,1
7,AE22CWTBLYAYLFFN75OH3JBDHW6A,"[B007ZF4FS6, B01H4E5DCY]","[4, 4]",B00C5NPGRW,5,B0042I2BWG,1


In [46]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

def calculate_baseline_metrics(df, id2idx, emb_matrix):
    pos_cosines = []
    neg_cosines = []

    for _, row in tqdm(df.iterrows()):
        hist_items = row['history']
        hist_ratings = row['ratings']

        embs, weights = [], []
        for item, r in zip(hist_items, hist_ratings):
            idx = id2idx.get(item)
            if idx is not None:
                embs.append(emb_matrix[idx])
                weights.append(float(r))

        if not embs:
            continue

        user_profile = np.average(embs, axis=0, weights=weights).reshape(1, -1)

        pos_idx = id2idx.get(row['positive_target'])
        neg_idx = id2idx.get(row['negative_target'])

        if pos_idx is None or neg_idx is None:
            continue

        pos_emb = emb_matrix[pos_idx].reshape(1, -1)
        neg_emb = emb_matrix[neg_idx].reshape(1, -1)

        cos_pos = float(cosine_similarity(user_profile, pos_emb)[0, 0])
        cos_neg = float(cosine_similarity(user_profile, neg_emb)[0, 0])

        pos_cosines.append(cos_pos)
        neg_cosines.append(cos_neg)

    gaps = np.array(pos_cosines) - np.array(neg_cosines)

    metrics = {
        "Mean Gap": np.mean(gaps),
        "Median Gap": np.median(gaps),
        "Gap > 0 Ratio": np.mean(gaps > 0),
        "Avg Cos Pos": np.mean(pos_cosines),
        "Avg Cos Neg": np.mean(neg_cosines)
    }
    return metrics

results = calculate_baseline_metrics(neg_feedback_df, id2idx, emb_matrix)

print("--- Baseline Evaluation Results ---")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

93557it [02:12, 705.10it/s]

--- Baseline Evaluation Results ---
Mean Gap: 0.0080
Median Gap: 0.0088
Gap > 0 Ratio: 0.5196
Avg Cos Pos: 0.4236
Avg Cos Neg: 0.4157


### Train

In [47]:
neg_feedback_df["history_idx"] = neg_feedback_df["history"].apply(lambda xs: [id2idx[x] for x in xs if x in id2idx])
neg_feedback_df["pos_idx"] = neg_feedback_df["positive_target"].map(id2idx)
neg_feedback_df["neg_idx"] = neg_feedback_df["negative_target"].map(id2idx)

In [48]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class PairDataset(Dataset):
    def __init__(self, df):
        self.histories = df["history_idx"].tolist()
        self.pos = df["pos_idx"].tolist()
        self.neg = df["neg_idx"].tolist()

    def __len__(self):
        return len(self.histories)

    def __getitem__(self, idx):
        return {
            "history_idx": self.histories[idx],
            "pos_idx": self.pos[idx],
            "neg_idx": self.neg[idx],
        }

def collate_fn(batch):
    B = len(batch)
    lengths = [len(x["history_idx"]) for x in batch]
    max_len = max(lengths)

    hist_idx = torch.full((B, max_len), -1, dtype=torch.long)
    mask = torch.zeros((B, max_len), dtype=torch.bool)
    pos_idx = torch.zeros(B, dtype=torch.long)
    neg_idx = torch.zeros(B, dtype=torch.long)

    for i, x in enumerate(batch):
        l = len(x["history_idx"])
        hist_idx[i, :l] = torch.tensor(x["history_idx"], dtype=torch.long)
        mask[i, :l] = True
        pos_idx[i] = x["pos_idx"]
        neg_idx[i] = x["neg_idx"]

    return {
        "hist_idx": hist_idx,
        "mask": mask,
        "pos_idx": pos_idx,
        "neg_idx": neg_idx,
    }

In [49]:
full_ds = PairDataset(neg_feedback_df)

n_total = len(full_ds)
n_val = int(0.1 * n_total)
n_train = n_total - n_val

train_ds, val_ds = random_split(
    full_ds,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, collate_fn=collate_fn, num_workers=2)

n_train, n_val

(84202, 9355)

In [50]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [90]:
emb_tensor = torch.tensor(np.array(emb_matrix), dtype=torch.float32)
emb_dim = emb_tensor.shape[1]

In [89]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionProfile(nn.Module):
    def __init__(self, emb_matrix, hidden_dim=128, mix_alpha=0.2):
        super().__init__()
        self.item_emb = nn.Embedding.from_pretrained(emb_matrix, freeze=True)
        d = emb_matrix.shape[1]
        self.mix_alpha = mix_alpha

        self.scorer = nn.Sequential(
            nn.Linear(d, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, hist_idx, mask):
        # hist_idx: [B, L]
        hist_emb = self.item_emb(hist_idx.clamp_min(0))          # [B, L, d]
        hist_emb = F.normalize(hist_emb, dim=-1)

        # baseline profile = mean trên embedding gốc
        masked_hist = hist_emb * mask.unsqueeze(-1)              # [B, L, d]
        denom = mask.sum(dim=1, keepdim=True).clamp_min(1)       # [B, 1]
        base_profile = masked_hist.sum(dim=1) / denom
        base_profile = F.normalize(base_profile, dim=-1)

        # attention profile = weighted sum trên embedding gốc
        logits = self.scorer(hist_emb).squeeze(-1)               # [B, L]
        logits = logits.masked_fill(~mask, -1e9)
        attn = torch.softmax(logits, dim=1)                      # [B, L]

        attn_profile = torch.sum(hist_emb * attn.unsqueeze(-1), dim=1)
        attn_profile = F.normalize(attn_profile, dim=-1)

        # mix nhẹ với baseline
        profile = (1 - self.mix_alpha) * base_profile + self.mix_alpha * attn_profile
        profile = F.normalize(profile, dim=-1)

        return profile, attn, base_profile, attn_profile

    def get_item_emb(self, item_idx):
        emb = self.item_emb(item_idx)
        emb = F.normalize(emb, dim=-1)
        return emb

In [94]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    all_pos = []
    all_neg = []
    all_gap = []
    all_drift = []

    for batch in tqdm(loader, leave=False):
        hist_idx = batch["hist_idx"].to(device)
        mask = batch["mask"].to(device)
        pos_idx = batch["pos_idx"].to(device)
        neg_idx = batch["neg_idx"].to(device)

        profile, attn, base_profile, attn_profile = model(hist_idx, mask)
        pos_emb = model.get_item_emb(pos_idx)
        neg_emb = model.get_item_emb(neg_idx)

        cos_pos = torch.sum(profile * pos_emb, dim=-1)
        cos_neg = torch.sum(profile * neg_emb, dim=-1)
        gap = cos_pos - cos_neg
        drift = 1 - torch.sum(profile * base_profile, dim=-1)

        all_pos.append(cos_pos.cpu())
        all_neg.append(cos_neg.cpu())
        all_gap.append(gap.cpu())
        all_drift.append(drift.cpu())

    all_pos = torch.cat(all_pos).numpy()
    all_neg = torch.cat(all_neg).numpy()
    all_gap = torch.cat(all_gap).numpy()
    all_drift = torch.cat(all_drift).numpy()

    return {
        "avg_cos_pos": float(all_pos.mean()),
        "avg_cos_neg": float(all_neg.mean()),
        "mean_gap": float(all_gap.mean()),
        "median_gap": float(np.median(all_gap)),
        "gap_gt_0": float((all_gap > 0).mean()),
        "avg_drift_from_base": float(all_drift.mean()),
    }

In [101]:
from tqdm import tqdm

def train_one_epoch(model, loader, optimizer, margin=0.02, lambda_anchor=0.1):
    model.train()
    total_loss = 0.0
    total_n = 0

    for batch in tqdm(loader):
        hist_idx = batch["hist_idx"].to(device)
        mask = batch["mask"].to(device)
        pos_idx = batch["pos_idx"].to(device)
        neg_idx = batch["neg_idx"].to(device)

        profile, attn, base_profile, attn_profile = model(hist_idx, mask)
        pos_emb = model.get_item_emb(pos_idx)
        neg_emb = model.get_item_emb(neg_idx)

        cos_pos = torch.sum(profile * pos_emb, dim=-1)
        cos_neg = torch.sum(profile * neg_emb, dim=-1)

        rank_loss = F.relu(margin - cos_pos + cos_neg).mean()
        anchor_loss = (1 - torch.sum(profile * base_profile, dim=-1)).mean()

        loss = rank_loss + lambda_anchor * anchor_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = hist_idx.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / total_n

In [161]:
model = AttentionProfile(emb_tensor, hidden_dim=128, mix_alpha=0.5).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

print("Before train:", evaluate(model, val_loader))

best_state = None
best_gap = -1e9

for epoch in range(10):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        margin=0.03,
        lambda_anchor=0.1
    )
    val_metrics = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train loss: {train_loss:.4f}")
    print(val_metrics)

    if val_metrics["mean_gap"] > best_gap:
        best_gap = val_metrics["mean_gap"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print("Best:", evaluate(model, val_loader))

Before train: {'avg_cos_pos': 0.42422592639923096, 'avg_cos_neg': 0.42660754919052124, 'mean_gap': -0.0023816144093871117, 'median_gap': 0.0014239251613616943, 'gap_gt_0': 0.502618920363442, 'avg_drift_from_base': 1.0458547876623925e-05}


100%|██████████| 329/329 [00:07<00:00, 43.99it/s]



Epoch 1
Train loss: 0.0888
{'avg_cos_pos': 0.4144275486469269, 'avg_cos_neg': 0.40616342425346375, 'mean_gap': 0.00826413556933403, 'median_gap': 0.00945371389389038, 'gap_gt_0': 0.521111704970604, 'avg_drift_from_base': 0.02746133878827095}


100%|██████████| 329/329 [00:05<00:00, 61.64it/s]



Epoch 2
Train loss: 0.0883
{'avg_cos_pos': 0.4120587110519409, 'avg_cos_neg': 0.4031784236431122, 'mean_gap': 0.008880308829247952, 'median_gap': 0.009260177612304688, 'gap_gt_0': 0.5215392838054517, 'avg_drift_from_base': 0.031470946967601776}


...

100%|██████████| 329/329 [00:07<00:00, 46.44it/s]



Epoch 9
Train loss: 0.0875
{'avg_cos_pos': 0.4110991358757019, 'avg_cos_neg': 0.4024270176887512, 'mean_gap': 0.008672087453305721, 'median_gap': 0.008415788412094116, 'gap_gt_0': 0.519722073757349, 'avg_drift_from_base': 0.032498542219400406}


100%|██████████| 329/329 [00:05<00:00, 63.34it/s]



Epoch 10
Train loss: 0.0874
{'avg_cos_pos': 0.41134703159332275, 'avg_cos_neg': 0.40255290269851685, 'mean_gap': 0.008794107474386692, 'median_gap': 0.009285390377044678, 'gap_gt_0': 0.519722073757349, 'avg_drift_from_base': 0.03311648592352867}


Best: {'avg_cos_pos': 0.4117020070552826, 'avg_cos_neg': 0.40272361040115356, 'mean_gap': 0.0089784050360322, 'median_gap': 0.009481668472290039, 'gap_gt_0': 0.5213254943880278, 'avg_drift_from_base': 0.03289133310317993}


In [163]:
from pprint import pprint

model.load_state_dict(best_state)
pprint(evaluate(model, val_loader))

{'avg_cos_neg': 0.40272361040115356,
 'avg_cos_pos': 0.4117020070552826,
 'avg_drift_from_base': 0.03289133310317993,
 'gap_gt_0': 0.5213254943880278,
 'mean_gap': 0.0089784050360322,
 'median_gap': 0.009481668472290039}


In [64]:
neg_feedback_val_df = neg_feedback_df.iloc[val_ds.indices].reset_index(drop=True)
calculate_baseline_metrics(neg_feedback_val_df, id2idx, emb_matrix)

9355it [00:13, 698.10it/s]


{'Mean Gap': np.float64(0.0025676793975842036),
 'Median Gap': np.float64(0.004708939975217041),
 'Gap > 0 Ratio': np.float64(0.5107429182255478),
 'Avg Cos Pos': np.float64(0.4206124832877505),
 'Avg Cos Neg': np.float64(0.41804480389016635)}

In [164]:
sample = test_df.iloc[40]
user_id = sample['user_id']
history = sample['history']

MAX_CONTEXT = 20

hist_indices = [id2idx[item] for item in history[:MAX_CONTEXT] if item in id2idx]
hist_embs = [emb_matrix[idx] for idx in hist_indices]

if not hist_embs:
    print("History items not found in embedding matrix.")
else:
    weights = np.ones(len(hist_embs))
    profile_weighted = np.average(hist_embs, axis=0, weights=weights).astype('float32')

    model.eval()
    with torch.no_grad():
        hist_tensor = torch.tensor([hist_indices]).to(device)
        mask_tensor = torch.ones_like(hist_tensor, dtype=torch.bool).to(device)
        profile_attn_ts, _, _, _ = model(hist_tensor, mask_tensor)
        profile_attn = profile_attn_ts.cpu().numpy()[0]

    recs_weighted = search_topk(index, ids, profile_weighted, topk=100)
    recs_attn = search_topk(index, ids, profile_attn, topk=100)

    set_weighted = set([r['product_id'] for r in recs_weighted])
    set_attn = set([r['product_id'] for r in recs_attn])

    intersection = set_weighted.intersection(set_attn)
    only_weighted = set_weighted - set_attn
    only_attn = set_attn - set_weighted

    print(f"User ID: {user_id}")
    print(f"History Length: {len(history)}")
    print("-" * 30)
    print(f"Số lượng sản phẩm giao nhau (Intersection): {len(intersection)}")
    print(f"Chỉ có ở Weighted Avg: {len(only_weighted)}")
    print(f"Chỉ có ở Attention: {len(only_attn)}")
    print("-" * 30)

    print("\nVí dụ 5 sản phẩm đầu tiên của Weighted Avg:")
    display(pd.DataFrame(recs_weighted[:5]))

    print("\nVí dụ 5 sản phẩm đầu tiên của Attention:")
    display(pd.DataFrame(recs_attn[:5]))

User ID: AFZUK3MTBIBEDQOPAK3OATUOUKLA
History Length: 188
------------------------------
Số lượng sản phẩm giao nhau (Intersection): 45
Chỉ có ở Weighted Avg: 55
Chỉ có ở Attention: 55
------------------------------

Ví dụ 5 sản phẩm đầu tiên của Weighted Avg:


,product_id,score
0,B004HGQMG4,0.793051
1,B07GT8YJK5,0.776664
2,B09TBB59KF,0.776329
3,B08BYR93YX,0.771737
4,B08BYQQQCJ,0.768904



Ví dụ 5 sản phẩm đầu tiên của Attention:


,product_id,score
0,B09TBB59KF,0.764272
1,B09YXTVYRZ,0.760862
2,B07MNT45J4,0.760842
3,B08BYR93YX,0.759720
4,B01FD47SFW,0.759387


In [165]:
MODEL_SAVE_PATH = "/content/drive/MyDrive/office_products_split/attention_profile_model.pth"

torch.save(best_state, MODEL_SAVE_PATH)

## Load and test


In [166]:
# 1. Khởi tạo model mới và load state_dict
loaded_model = AttentionProfile(emb_tensor, hidden_dim=128, mix_alpha=0.5).to(device)
loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
loaded_model.eval()

print(f"Đã load mô hình từ: {MODEL_SAVE_PATH}")

# 2. Test thử với 1 sample từ test_df
sample = test_df.iloc[42]  # Lấy sample khác để kiểm tra
history = sample['history'][:MAX_CONTEXT]
hist_indices = [id2idx[item] for item in history if item in id2idx]

if not hist_indices:
    print("Không tìm thấy item trong history.")
else:
    with torch.no_grad():
        hist_tensor = torch.tensor([hist_indices]).to(device)
        mask_tensor = torch.ones_like(hist_tensor, dtype=torch.bool).to(device)

        # Lấy profile từ mô hình vừa load
        profile_ts, _, _, _ = loaded_model(hist_tensor, mask_tensor)
        profile_vec = profile_ts.cpu().numpy()[0]

    # 3. Tìm kiếm top 5 sản phẩm
    test_recs = search_topk(index, ids, profile_vec, topk=5)

    print(f"User ID: {sample['user_id']}")
    print("Top 5 gợi ý từ mô hình vừa load:")
    display(pd.DataFrame(test_recs))

Đã load mô hình từ: /content/drive/MyDrive/office_products_split/attention_profile_model.pth
User ID: AFZUK3MTBIBEDQOPAK3OATUOUKLA
Top 5 gợi ý từ mô hình vừa load:


,product_id,score
0,B09TBB59KF,0.764272
1,B09YXTVYRZ,0.760862
2,B07MNT45J4,0.760842
3,B08BYR93YX,0.759720
4,B01FD47SFW,0.759387
